In [ ]:
#pip install --upgrade gradio

In [ ]:
import gradio as gr
import numpy as np
import soundfile as sf
import tempfile
import os
import librosa
from preprocessing import preprocess_recording_to_batch

In [ ]:


def audio_to_wav_path(audio):
    """
         standardizes that into a .wav file path on disk.
    """
    if audio is None:
        raise ValueError("No audio provided.")

    #  audio is already a filepath
    if isinstance(audio, str) and os.path.exists(audio):
        return audio

    # audio is (sr, y)
    if isinstance(audio, (tuple, list)) and len(audio) == 2:
        sr, y = audio
        y = np.asarray(y)


        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        tmp.close()
        sf.write(tmp.name, y, sr)
        return tmp.name

    raise ValueError(f"Unsupported audio format from Gradio: {type(audio)}")



In [ ]:
def preprocess_in_ram(audio):
    """

      audio (gradio) -> wav path -> preprocess -> spectrogram batch (numpy)
    
    returns:
      batch_np -> goes into gr.State (spec_state)
      wav_path -> goes into gr.State (wav_state) 
      status string -> goes into status textbox
    """
    wav_path = audio_to_wav_path(audio)

    batch = preprocess_recording_to_batch(wav_path, return_torch=True)

    if hasattr(batch, "detach"):
        batch_np = batch.detach().cpu().numpy().astype(np.float32)
    else:
        batch_np = np.asarray(batch, dtype=np.float32)

    if batch_np.ndim != 4 or batch_np.shape[1] != 1:
        raise ValueError(f"Unexpected batch shape: {batch_np.shape}. Expected [N, 1, 80, 300].")

    if batch_np.shape[0] == 0:
        return batch_np, wav_path, "No valid audio after preprocessing (too much silence or too short)."

    return (
        batch_np,
        wav_path,
        "Preprocessing complete\n"
        f"- Path (temp or upload): {wav_path}\n"
        f"- Batch shape: {batch_np.shape}  (N, 1, 80, 300)\n"
        f"- Ready to feed into the model"
    )

In [ ]:
import numpy as np
import torch

from model import ResNet18Mel


CKPT_PATH = "best_model.pth"  #  same directory as notebook

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(CKPT_PATH, map_location=device)


state_dict = ckpt["model_state"]


class_to_label = ckpt.get("class_to_label", {0: "class_0", 1: "class_1"})
num_classes = len(class_to_label)

model = ResNet18Mel(pretrained=False).to(device)
model.load_state_dict(state_dict)
model.eval()


expected_shape = ckpt.get("img_shape", None)  

print("Model loaded")
print(" - device:", device)
print(" - num_classes:", num_classes)
print(" - class_to_label:", class_to_label)
print(" - expected img_shape:", expected_shape)


In [ ]:
def aggregate_probs_mean(probs_np: np.ndarray) -> np.ndarray:
    """
    probs_np: [N, C] softmax probabilities per chunk
    returns:  [C] mean probability
    """
    return probs_np.mean(axis=0)

In [ ]:
def run_model_from_state(spec_batch: np.ndarray) -> str:
    """
    Takes spectrogram batch from spec_state (numpy [N,1,80,300]) and runs model.
    Returns a status string for your Gradio UI.
    """
    if spec_batch is None:
        return "Nothing preprocessed yet. Click Preprocess first."


    if not isinstance(spec_batch, np.ndarray):
        spec_batch = np.asarray(spec_batch, dtype=np.float32)


    if spec_batch.ndim != 4:
        return f"Unexpected spectrogram shape: {spec_batch.shape}. Expected [N, 1, 80, 300]."
    if spec_batch.shape[0] == 0:
        return "No valid chunks (audio too short / too much silence)."
    
    if expected_shape is not None:
        got = tuple(spec_batch.shape[1:])  
        exp = tuple(expected_shape)  

        ok = False

        #checkpoint includes channel
        if got == exp:
            ok = True

        #  checkpoint excludes channel
        elif got[1:] == exp:
            ok = True

        if not ok:
            return (
                "Shape mismatch vs training\n"
                f"- got (C,H,W): {got}\n"
                f"- expected: {exp} (checkpoint)\n"
                "Tip: checkpoint img_shape may omit the channel dimension."
            )


    x = torch.from_numpy(spec_batch).float().to(device)

    with torch.no_grad():
        logits = model(x)                
        probs = torch.softmax(logits, dim=1)  

    probs_np = probs.detach().cpu().numpy() 

    # aggregates across chunks
    agg = aggregate_probs_mean(probs_np)   
    pred_class = int(np.argmax(agg))
    pred_label = class_to_label.get(pred_class, str(pred_class))
    confidence = float(agg[pred_class])


    lines = []
    lines.append("Prediction complete")
    lines.append(f"- Chunks: {spec_batch.shape[0]}")
    lines.append(f"- Predicted class: {pred_class} ({pred_label})")
    lines.append(f"- Confidence (mean prob): {confidence:.4f}")
    lines.append("")
    lines.append("Per-class mean probabilities:")
    for cls_idx in range(num_classes):
        lbl = class_to_label.get(cls_idx, str(cls_idx))
        lines.append(f"  - {cls_idx} ({lbl}): {float(agg[cls_idx]):.4f}")

    # show per-chunk confidence for the predicted class
    lines.append("")
    lines.append(f"Per-chunk prob for predicted class ({pred_label}):")
    chunk_probs = probs_np[:, pred_class]
    for i, p in enumerate(chunk_probs):
        lines.append(f"  - chunk {i:02d}: {float(p):.4f}")

    return "\n".join(lines)

In [ ]:
import numpy as np

def run_model_for_ui(spec_batch: np.ndarray):
    """
    returns:
      - result_box (Markdown): big friendly message
      - status (Textbox): compact debug info
    """
    if spec_batch is None:
        return (
            "## No spectrograms yet\nClick **Preprocess** first.",
            "Nothing preprocessed yet."
        )

    if not isinstance(spec_batch, np.ndarray):
        spec_batch = np.asarray(spec_batch, dtype=np.float32)

    if spec_batch.ndim != 4 or spec_batch.shape[0] == 0:
        return (
            "## Invalid input\nAudio may be too short or mostly silence.",
            f"Bad batch shape: {spec_batch.shape}"
        )


    x = torch.from_numpy(spec_batch).float().to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)

    probs_np = probs.detach().cpu().numpy()  # [N, C]
    agg = probs_np.mean(axis=0)              # [C]
    pred_class = int(np.argmax(agg))
    confidence = float(agg[pred_class])



    pred_label = class_to_label.get(pred_class, str(pred_class))

    if pred_class == 1:
        headline = "## Result: **Person is Allowed**"
    else:
        headline = "## Result: **Person is NOT allowed**"

    result_md = (
        f"{headline}\n\n"
        f"**Confidence:** `{confidence:.4f}`\n\n"
    )

  
    status_lines = []
    status_lines.append("Prediction complete")
    status_lines.append(f"Chunks: {spec_batch.shape[0]}")
    status_lines.append(f"Predicted class: {pred_class} ({pred_label})")
    status_lines.append(f"Mean prob confidence: {confidence:.4f}")
    status_lines.append("Per-class mean probs:")
    for i in range(len(agg)):
        lbl = class_to_label.get(i, str(i))
        status_lines.append(f"  - {i} ({lbl}): {float(agg[i]):.4f}")

    return result_md, "\n".join(status_lines)


In [ ]:
import gradio as gr

with gr.Blocks(title="Voice Model GUI") as demo:

    # states
    spec_state = gr.State(None)
    wav_state  = gr.State(None)

    gr.Markdown("# Voice Recognition Model")
    gr.Markdown("Upload/record audio (3–15 seconds), preprocess it, then run the model.")

    with gr.Row():
        # main area
        with gr.Column(scale=3):
            audio_in = gr.Audio(
                sources=["upload", "microphone"],
                type="numpy",
                label="Input audio",
            )

            with gr.Row():
                prep_btn  = gr.Button("Preprocess", variant="primary")
                run_btn   = gr.Button("Run Model", variant="primary")
                clear_btn = gr.Button("Clear")

    
            result_box = gr.Markdown(
                value="### Result will appear here.",
                elem_id="result_box"
            )

        # side area
        with gr.Column(scale=1, min_width=260):
            # instructions
            instructions = gr.Markdown(
                """
## Step by step use:
1. **Upload** an audio file or **record** with the microphone.  
2. Click **Preprocess** to spectrogram chunks.  
3. Click **Run Model** to get the prediction.  
4. Use **Clear** to reset and try another audio.

## Tips
- Keep audio between **3–15 seconds**.
- Try speaking clearly and avoid loud background noise.
                """,
                elem_id="instructions_box"
            )

           
            status = gr.Textbox(
                label="Status (debug)",
                lines=8,
                max_lines=10,
            )


    prep_btn.click(
        fn=preprocess_in_ram,
        inputs=[audio_in],
        outputs=[spec_state, wav_state, status],
    )

    run_btn.click(
        fn=run_model_for_ui,       
        inputs=[spec_state],
        outputs=[result_box, status], 
    )

    clear_btn.click(
        fn=lambda: (None, None, "## Result will appear here.", "", None),
        inputs=[],
        outputs=[spec_state, wav_state, result_box, status, audio_in],
    )

demo


In [ ]:
demo.launch(share=False)

